In [31]:
import regex as re
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
def get_stats(L):
    S={(L[i],L[i+1]) for i in range(len(L)-1)}
    pair_count={s:0 for s in S}
    for i in range(len(L)-1):
        pair_count[L[i],L[i+1]]+=1
    return (pair_count)
def replace(ids,pair,idr):
    newids=[]
    i=0
    while i<len(ids):
        if i<len(ids)-1 and (ids[i],ids[i+1])==pair:
            newids.append(idr)
            i+=2
        else:
            newids.append(ids[i])
            i+=1
    return(newids)
class RegexTokenizer:
    def __init__(self,text,vocab_size,pattern=GPT4_SPLIT_PATTERN):
        self.pattern=pattern
        self.merges={}
    def train(self,text,vocab_size):
        num_merges=vocab_size-256
        chunks=re.findall(self.pattern, text)
        ids=[list(chunk.encode("utf_8"))for chunk in chunks]
        merges={}
        for i in range(num_merges):
            stats={}
            for chunk_ids in ids:
                chunk_stats=get_stats(chunk_ids)
                for s_chunk in chunk_stats:
                    stats[s_chunk] = stats.get(s_chunk, 0) + chunk_stats[s_chunk]
            rep_pair=max(stats,key=lambda x:stats[x])
            idr,new_ids=256+i,[]
            for chunk_ids in ids:
                new_ids.append(replace(chunk_ids,rep_pair,idr))
            ids=new_ids        
            merges[idr]=rep_pair
        self.merges=merges
    def _encode_chunks(self,chunk_ids):
        merges_inver={self.merges[i]:i for i in self.merges}
        changed = True
        while changed:
            changed = False
            for pair, new_id in merges_inver.items():
                new_ids = replace(chunk_ids, pair, new_id)
                if new_ids != chunk_ids:
                    chunk_ids = new_ids
                    changed = True
        return chunk_ids
 
    def encode(self,text):
        merges_inver={self.merges[i]:i for i in self.merges}
        chunks=re.findall(self.pattern, text)
        ids=[list(chunk.encode("utf_8"))for chunk in chunks]
        new_ids = [self._encode_chunks(chunk_ids) for chunk_ids in ids]
        flat_ids=[]
        for i in new_ids:
            flat_ids+=i
        return(flat_ids)
    def decode(self,ids):
        proof=1
        while proof==1:
            proof=0
            new_list=[]
            for token in ids:
                if token in self.merges:
                    new_list.append(self.merges[token][0])
                    new_list.append(self.merges[token][1])
                    proof=1
                else:
                        new_list.append(token)
            ids=new_list
        text = bytes(ids).decode('utf-8',errors="replace")
        return(text)  
